# 74 ? Experimento TF-IDF + HDBSCAN

Este notebook agrupa descri??es normalizadas de itens de supermercado usando TF-IDF e HDBSCAN. A an?lise usa dist?ncia cosseno, n?o usa NCM nem r?tulos humanos e trata ru?do como resultado expl?cito do algoritmo. A amostra configur?vel mant?m o custo do experimento controlado. Execute de cima para baixo em uma sess?o limpa.


In [1]:
from sklearn import config_context
from hashlib import sha256
from datetime import datetime, timezone
from pathlib import Path
import logging
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from matplotlib.colors import ListedColormap

from nofis_classifier.schema import DESCRIPTION_NORMALIZED, FREQUENCY

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("notebook74")
pd.set_option("display.max_colwidth", 140)


## Configuracoes e comparabilidade

Controles comuns aos notebooks 70 a 76:

- Base completa `2025_NCMs_filtrados_semstopwords`, periodo 202501-202512.
- Ate 20.000 descricoes unicas para agrupamento, amostradas com semente 42 e ordenadas por descricao normalizada. Frequencia nao pondera o ajuste.
- Silhouette cosseno sobre uma amostra base fixa de ate 20.000 linhas, com semente 42. Ruido do algoritmo e excluido depois da amostragem; a quantidade efetivamente avaliada e registrada.
- Ranking exige silhouette definida e de 10 a 1.000 clusters efetivamente formados, sem contar ruido. Pesos: 0,75 para silhouette reescalada e 0,25 para cobertura do algoritmo.
- t-SNE 2D e 3D: mesmos ate 5.000 itens, semente 42, TruncatedSVD de ate 50 componentes, perplexidade 30, 1.000 iteracoes, init PCA (quando ha dimensoes suficientes), learning_rate auto e Barnes-Hut. As projecoes de representacoes diferentes sao exploratorias, nao um espaco comum de distancias.
- `comparison` em `run.json` registra os controles e o hash das descricoes/frequencias usadas no agrupamento.

TF-IDF: os mesmos 10 analyzers dos experimentos de texto, cruzados com `MIN_DF_VALUES = [1, 2, 5, 10]`, max_features 100.000, max_df 1.0, sublinear_tf True, norm l2 e dtype float64. Os vetores zerados sao registrados como diagnostico da representacao, separados do ruido do algoritmo.

HDBSCAN: min_cluster_size 2/5/10/20, min_samples None/2/5/10, selecao eom, metric cosine, algorithm brute e n_jobs 1, como nos notebooks 75 e 76. Grade atual: 40 representacoes x 16 combinacoes = 640 ajustes.


In [2]:
DATASET_NAME = "2025_NCMs_filtrados_semstopwords"

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("N?o encontrei a raiz do reposit?rio com data/processed/.")
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / DATASET_NAME
DATASET_FILES = sorted(DATASET_DIR.glob("*.parquet"))
if not DATASET_FILES:
    raise FileNotFoundError(f"Nenhum Parquet encontrado para {DATASET_NAME}: {DATASET_DIR}")
if len(DATASET_FILES) > 1:
    raise ValueError(
        f"Mais de um Parquet encontrado em {DATASET_DIR}; selecione o arquivo/per?odo explicitamente: "
        f"{[path.name for path in DATASET_FILES]}"
    )
DATASET_PATH = DATASET_FILES[0]
if not DATASET_PATH.stem.startswith(f"{DATASET_NAME}-"):
    raise ValueError(
        f"O nome do arquivo deve come?ar com '{DATASET_NAME}-' para derivar o per?odo: "
        f"{DATASET_PATH.name}"
    )

PERIOD_LABEL = DATASET_PATH.stem[len(DATASET_NAME) + 1:]
MAX_DESCRIPTIONS = 20_000
SILHOUETTE_SAMPLE_SIZE = 20_000
SAMPLE_SEEDS = [42]
ANALYZER_CONFIGS = {'char_wb_2_4': ('char_wb', (2, 4)), 'char_wb_2_10': ('char_wb', (2, 10)), 'char_wb_3_5': ('char_wb', (3, 5)), 'char_wb_3_10': ('char_wb', (3, 10)), 'char_wb_4_8': ('char_wb', (4, 8)), 'char_wb_4_10': ('char_wb', (4, 10)), 'char_wb_5_10': ('char_wb', (5, 10)), 'word_1_3': ('word', (1, 3)), 'word_1_5': ('word', (1, 5)), 'word_2_5': ('word', (2, 5))}
MIN_DF_VALUES = [1, 2, 5, 10]
MAX_FEATURES = 100_000
MIN_CLUSTER_SIZE_VALUES = [2, 5, 10, 20]
MIN_SAMPLES_VALUES = [None, 2, 5, 10]
CLUSTER_SELECTION_METHOD = "eom"
HDBSCAN_N_JOBS = 1

OUTPUT_ROOT = PROJECT_ROOT / "reports/tfidf_hdbscan"
MODEL_ROOT = PROJECT_ROOT / "models/tfidf_hdbscan"


TSNE_MAX_POINTS = 5_000
TSNE_PERPLEXITY = 30
TSNE_MAX_ITER = 1_000
TSNE_RANDOM_STATE = 42
TSNE_3D_N_COMPONENTS = 3
TSNE_SVD_COMPONENTS = 50

METRIC_RANDOM_STATE = 42

PAIRWISE_WORKING_MEMORY_MIB = 64

MAX_DF = 1.0

ASSIGNMENT_RUN_ID = None

MODEL_RUN_ID = None


### Configura??o do ranking final

`MIN_CLUSTER_FOR_RANKING = 10` define o minimo do ranking. O intervalo ate `MAX_CLUSTERS_FOR_RANKING` inclui os dois extremos e nao altera os ajustes da grade. Em DBSCAN/HDBSCAN, a contagem exclui o grupo de ruido (-1).


In [3]:
# O limite filtra apenas o ranking e a sele??o final; n?o altera os ajustes do HDBSCAN.
MAX_CLUSTERS_FOR_RANKING = 1000
MIN_CLUSTER_FOR_RANKING = 10

# Os pesos definem a contribui??o relativa de silhouette e cobertura (1 - ru?do).
SILHOUETTE_WEIGHT = 0.75
NOISE_WEIGHT = 0.25


## 2. Fun??es da an?lise

A prepara??o remove descri??es vazias, mant?m uma linha por descri??o e valida que a frequ?ncia total n?o foi alterada. Cada configura??o ? executada sobre a mesma matriz TF-IDF da respectiva representa??o.


In [4]:
def prepare_items(frame):
    initial_frequency = frame[FREQUENCY].sum()
    prepared = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].copy()
    prepared[DESCRIPTION_NORMALIZED] = prepared[DESCRIPTION_NORMALIZED].astype("string").str.strip()
    prepared = prepared[prepared[DESCRIPTION_NORMALIZED].notna() & prepared[DESCRIPTION_NORMALIZED].ne("")]
    prepared = prepared[prepared[FREQUENCY] >= 1].copy()
    prepared = (
        prepared.groupby(DESCRIPTION_NORMALIZED, as_index=False, sort=False)[FREQUENCY]
        .sum()
    )
    if prepared[FREQUENCY].sum() != initial_frequency:
        raise ValueError("A prepara??o alterou a soma de frequ?ncia.")
    return prepared.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)

def summarize_clusters(items, labels):
    result = items.copy()
    result["cluster"] = labels
    result["ruido"] = result["cluster"].eq(-1)
    summary = (
        result.groupby("cluster", as_index=False)
        .agg(descricoes_unicas=(DESCRIPTION_NORMALIZED, "size"), frequencia_total=(FREQUENCY, "sum"))
        .sort_values("descricoes_unicas", ascending=False)
    )
    return result, summary

def score_configuration(matrix, labels, sample_size, random_state):
    """Avalia a mesma amostra de linhas, excluindo ruido somente apos amostrar."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_count = len(np.unique(labels[~noise_mask]))
    base_count = min(sample_size, len(labels))
    rng = np.random.default_rng(random_state)
    positions = np.sort(rng.choice(len(labels), size=base_count, replace=False))
    positions = positions[~noise_mask[positions]]
    sample_labels = labels[positions]
    sample_cluster_count = len(np.unique(sample_labels))
    score = np.nan
    actual_count = len(positions)
    if 2 <= sample_cluster_count < actual_count:
        with config_context(working_memory=PAIRWISE_WORKING_MEMORY_MIB):
            score = float(silhouette_score(matrix[positions], sample_labels, metric="cosine"))
    else:
        actual_count = 0
    return {
        "cluster_count_excluding_noise": cluster_count,
        "noise_count": int(noise_mask.sum()),
        "noise_fraction": float(noise_mask.mean()),
        "silhouette_base_sample_size": base_count,
        "silhouette_sample_size": actual_count,
        "silhouette_cosine_without_noise": score,
    }

def compute_tsne_projection(features, max_points, perplexity, max_iter, random_state, n_components=2):
    """Usa a mesma amostra, SVD e parametros t-SNE em todas as representacoes."""
    point_count = features.shape[0]
    sample_count = min(point_count, max_points)
    if sample_count < 3 or n_components not in (2, 3):
        raise ValueError("t-SNE exige pelo menos tres itens e n_components 2 ou 3.")
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(point_count, size=sample_count, replace=False))
    sampled = features[indices]
    dimensions = min(TSNE_SVD_COMPONENTS, sampled.shape[1] - 1, sample_count - 1)
    if dimensions >= 1:
        reduced = TruncatedSVD(n_components=dimensions, random_state=random_state).fit_transform(sampled)
    else:
        reduced = sampled.toarray() if hasattr(sampled, "toarray") else np.asarray(sampled)
        dimensions = reduced.shape[1]
    coordinates = TSNE(
        n_components=n_components,
        perplexity=min(float(perplexity), (sample_count - 1) / 3),
        init="pca" if dimensions >= n_components else "random",
        learning_rate="auto", max_iter=max_iter, random_state=random_state,
        method="barnes_hut",
    ).fit_transform(reduced)
    return indices, coordinates, dimensions


def save_tsne_plot(coordinates, labels, run_id, output_path):
    """Salva um PNG t-SNE colorido pelos clusters, com o ru?do em cinza."""
    labels = np.asarray(labels)
    noise_mask = labels == -1
    cluster_ids = np.unique(labels[~noise_mask])
    if cluster_ids.size:
        palette = plt.get_cmap("tab20").colors
        cluster_cmap = ListedColormap([
            palette[index % len(palette)] for index in range(len(cluster_ids))
        ])
        cluster_codes = np.searchsorted(cluster_ids, labels[~noise_mask])
    else:
        cluster_cmap = None
        cluster_codes = np.empty(0, dtype=int)

    figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
    if noise_mask.any():
        axis.scatter(
            coordinates[noise_mask, 0], coordinates[noise_mask, 1],
            c="#b8b8b8", s=8, alpha=0.3, linewidths=0,
        )
    if cluster_ids.size:
        axis.scatter(
            coordinates[~noise_mask, 0], coordinates[~noise_mask, 1],
            c=cluster_codes, cmap=cluster_cmap, vmin=-0.5,
            vmax=len(cluster_ids) - 0.5, s=9, alpha=0.75, linewidths=0,
        )
    axis.set_title(
        f"t-SNE: {run_id}\n{len(cluster_ids)} clusters; "
        f"{int(noise_mask.sum()):,} descri??es como ru?do"
    )
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.text(
        0.01, 0.01,
        "Ru?do em cinza; as cores se repetem a cada 20 clusters.",
        transform=axis.transAxes, fontsize=8, color="#555555",
    )
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def save_tsne_3d_html(coordinates, labels, run_id, output_path):
    """Salva uma visualiza??o t-SNE 3D interativa e autocontida em HTML."""
    points = [
        {"x": float(x), "y": float(y), "z": float(z), "label": int(label)}
        for (x, y, z), label in zip(coordinates, labels)
    ]
    data_json = json.dumps(points, separators=(",", ":"))
    title = str(run_id).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    page = r"""<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>t-SNE 3D: __TITLE__</title>
<style>
html, body { margin: 0; height: 100%; background: #f7f8fa; color: #202124; font: 14px system-ui, sans-serif; }
main { display: flex; flex-direction: column; height: 100%; }
header { padding: 12px 18px; background: white; border-bottom: 1px solid #ddd; }
h1 { margin: 0 0 4px; font-size: 17px; }
p { margin: 0; color: #555; }
canvas { display: block; width: 100%; flex: 1; min-height: 300px; cursor: grab; }
canvas:active { cursor: grabbing; }
</style>
</head>
<body>
<main>
<header><h1>Proje??o t-SNE 3D: __TITLE__</h1><p>Arraste para girar; use a roda do mouse para aproximar. Ru?do em cinza.</p></header>
<canvas id="plot"></canvas>
</main>
<script>
const points = __POINTS__;
const canvas = document.getElementById("plot");
const ctx = canvas.getContext("2d");
let yaw = 0.65, pitch = -0.35, zoom = 1, dragging = false, lastX = 0, lastY = 0;
const colors = ["#3366cc", "#dc3912", "#ff9900", "#109618", "#990099", "#0099c6", "#dd4477", "#66aa00", "#b82e2e", "#316395", "#994499", "#22aa99", "#aaaa11", "#6633cc", "#e67300", "#8b0707", "#651067", "#329262", "#5574a6", "#3b3eac"];
const bounds = ["x", "y", "z"].map(key => {
  const values = points.map(point => point[key]);
  const low = Math.min(...values), high = Math.max(...values);
  return { low, span: Math.max(high - low, 1e-9) };
});
function resize() {
  const ratio = window.devicePixelRatio || 1;
  canvas.width = Math.max(1, Math.floor(canvas.clientWidth * ratio));
  canvas.height = Math.max(1, Math.floor(canvas.clientHeight * ratio));
  ctx.setTransform(ratio, 0, 0, ratio, 0, 0);
  draw();
}
function draw() {
  const width = canvas.clientWidth, height = canvas.clientHeight;
  ctx.clearRect(0, 0, width, height);
  const scale = Math.min(width, height) * 0.7 * zoom;
  const cy = Math.cos(yaw), sy = Math.sin(yaw), cp = Math.cos(pitch), sp = Math.sin(pitch);
  const projected = points.map(point => {
    const x = ((point.x - bounds[0].low) / bounds[0].span - 0.5) * 2;
    const y = ((point.y - bounds[1].low) / bounds[1].span - 0.5) * 2;
    const z = ((point.z - bounds[2].low) / bounds[2].span - 0.5) * 2;
    const rx = x * cy - z * sy, rz = x * sy + z * cy;
    const ry = y * cp - rz * sp, depth = y * sp + rz * cp;
    return { x: width / 2 + rx * scale / 2, y: height / 2 - ry * scale / 2, depth, label: point.label };
  }).sort((a, b) => a.depth - b.depth);
  for (const point of projected) {
    ctx.globalAlpha = point.label === -1 ? 0.24 : 0.78;
    ctx.fillStyle = point.label === -1 ? "#999999" : colors[Math.abs(point.label) % colors.length];
    ctx.beginPath();
    ctx.arc(point.x, point.y, point.label === -1 ? 2 : 3, 0, Math.PI * 2);
    ctx.fill();
  }
  ctx.globalAlpha = 1;
}
canvas.addEventListener("pointerdown", event => { dragging = true; lastX = event.clientX; lastY = event.clientY; canvas.setPointerCapture(event.pointerId); });
canvas.addEventListener("pointerup", () => { dragging = false; });
canvas.addEventListener("pointermove", event => {
  if (!dragging) return;
  yaw += (event.clientX - lastX) * 0.008;
  pitch = Math.max(-1.5, Math.min(1.5, pitch + (event.clientY - lastY) * 0.008));
  lastX = event.clientX; lastY = event.clientY; draw();
});
canvas.addEventListener("wheel", event => { event.preventDefault(); zoom = Math.max(0.3, Math.min(5, zoom * (event.deltaY < 0 ? 1.1 : 0.9))); draw(); }, { passive: false });
window.addEventListener("resize", resize);
resize();
</script>
</body>
</html>"""
    page = page.replace("__TITLE__", title).replace("__POINTS__", data_json)
    output_path.write_text(page, encoding="utf-8")


def comparison_metadata(frame):
    """Registra controles comuns e a identidade exata da amostra de agrupamento."""
    sample = frame[[DESCRIPTION_NORMALIZED, FREQUENCY]].sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    return {
        "dataset_name": DATASET_NAME,
        "training_description_count": len(sample),
        "training_sample_sha256": sha256(pd.util.hash_pandas_object(sample, index=False).values.tobytes()).hexdigest(),
        "max_descriptions": MAX_DESCRIPTIONS, "sample_seed": 42,
        "frequency_weighted": False,
        "silhouette_metric": "cosine", "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
        "metric_random_state": METRIC_RANDOM_STATE,
        "metric_sampling": "fixed rows, then exclude algorithmic noise",
        "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
        "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
        "silhouette_weight": SILHOUETTE_WEIGHT, "noise_weight": NOISE_WEIGHT,
        "tsne": {
            "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
            "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
            "reduction": "TruncatedSVD", "svd_components": TSNE_SVD_COMPONENTS,
            "dimensions": [2, 3], "init": "pca", "learning_rate": "auto",
            "method": "barnes_hut",
        },
    }


## 3. Carregar e preparar dados

Origem: Parquet produzido pelos notebooks de prepara??o de dados do projeto, com descri??es de supermercado, NCMs e duplicatas agregadas por descri??o. A amostragem ? feita uma vez por semente e reutilizada entre os analyzers daquela semente.


In [5]:
raw_items = pd.read_parquet(DATASET_PATH)
required_columns = {DESCRIPTION_NORMALIZED, FREQUENCY}
missing_columns = required_columns - set(raw_items.columns)
if missing_columns:
    raise ValueError(f"Colunas ausentes no dataset: {sorted(missing_columns)}")
base_items = prepare_items(raw_items)
logger.info("Descricoes disponiveis apos preparacao: %d", len(base_items))


2026-10-25 01:11:04,111 INFO Descricoes disponiveis apos preparacao: 146999


## 4. Vetorizar, ajustar HDBSCAN e comparar par?metros

Cada combina??o de semente e analyzer gera uma matriz TF-IDF pr?pria. `metric="cosine"` trabalha diretamente com a matriz esparsa. O ru?do (`cluster = -1`) ? exclu?do da silhouette e contabilizado separadamente; configura??es com menos de dois clusters v?lidos ficam sem silhouette. Para controlar o custo, a silhouette usa no m?ximo `SILHOUETTE_SAMPLE_SIZE` itens n?o classificados como ru?do.

Para visualiza??o, uma amostra fixa de at? `TSNE_MAX_POINTS` descri??es ? reduzida com TruncatedSVD e projetada uma vez por representa??o. As coordenadas s?o reutilizadas para salvar um PNG por configura??o.

A proje??o t-SNE 3D ? calculada separadamente e cada configura??o tamb?m ? salva como HTML interativo autocontido em `tsne_3d_html/`.


In [6]:
records = []
labels_by_run = {}
items_by_run = {}
matrices_by_representation = {}
items_by_seed = {}

for sample_seed in SAMPLE_SEEDS:
    sampled_items = base_items
    if MAX_DESCRIPTIONS is not None and len(base_items) > MAX_DESCRIPTIONS:
        sampled_items = base_items.sample(n=MAX_DESCRIPTIONS, random_state=sample_seed)
    sampled_items = sampled_items.sort_values(DESCRIPTION_NORMALIZED).reset_index(drop=True)
    items_by_seed[sample_seed] = sampled_items

    for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items():
        for min_df in MIN_DF_VALUES:
            logger.info(
                "Vetorizando analyzer=%s, min_df=%d, seed=%d, descricoes=%d",
                analyzer, min_df, sample_seed, len(sampled_items),
            )
            vectorizer = TfidfVectorizer(
                analyzer=analyzer, ngram_range=ngram_range, min_df=min_df,
                max_features=MAX_FEATURES, max_df=MAX_DF, sublinear_tf=True, norm="l2", dtype=np.float64,
            )
            matrix = vectorizer.fit_transform(sampled_items[DESCRIPTION_NORMALIZED].astype(str))
            zero_vector_count = int((matrix.getnnz(axis=1) == 0).sum())
            matrices_by_representation[(sample_seed, representation, min_df)] = matrix
            logger.info(
                "Analyzer=%s min_df=%d seed=%d: %d termos, %d vetores zerados",
                analyzer, min_df, sample_seed, matrix.shape[1], zero_vector_count,
            )

            for min_cluster_size in MIN_CLUSTER_SIZE_VALUES:
                for min_samples in MIN_SAMPLES_VALUES:
                    min_samples_label = "default" if min_samples is None else str(min_samples)
                    run_id = (
                        f"seed{sample_seed}_{representation}_min_df{min_df}_min_cluster_size{min_cluster_size}"
                        f"_min_samples{min_samples_label}_{CLUSTER_SELECTION_METHOD}"
                    )
                    logger.info("Ajustando HDBSCAN %s", run_id)
                    model = HDBSCAN(
                        min_cluster_size=min_cluster_size,
                        min_samples=min_samples,
                        metric="cosine",
                        algorithm="brute",
                        cluster_selection_method=CLUSTER_SELECTION_METHOD,
                        n_jobs=HDBSCAN_N_JOBS,
                        copy=True,
                    )
                    labels = model.fit_predict(matrix)
                    metrics = score_configuration(
                        matrix, labels, SILHOUETTE_SAMPLE_SIZE,
                        METRIC_RANDOM_STATE,
                    )
                    records.append({
                        "run_id": run_id, "sample_seed": sample_seed,
                        "representation": representation, "analyzer": analyzer,
                        "ngram_range": str(ngram_range), "min_df": min_df,
                        "feature_count": matrix.shape[1], "zero_vector_count": zero_vector_count,
                        "min_cluster_size": min_cluster_size,
                        "min_samples": min_samples,
                        "cluster_selection_method": CLUSTER_SELECTION_METHOD,
                        "frequency_weighted": False, **metrics,
                    })
                    labels_by_run[run_id] = labels
                    items_by_run[run_id] = sampled_items

results = pd.DataFrame(records)


2026-10-25 01:11:04,150 INFO Vetorizando analyzer=char_wb, min_df=1, seed=42, descricoes=20000


2026-10-25 01:11:04,894 INFO Analyzer=char_wb min_df=1 seed=42: 36881 termos, 0 vetores zerados


2026-10-25 01:11:04,896 INFO Ajustando HDBSCAN seed42_char_wb_2_4_min_df1_min_cluster_size2_min_samplesdefault_eom


2026-10-25 01:11:17,850 INFO Ajustando HDBSCAN seed42_char_wb_2_4_min_df1_min_cluster_size2_min_samples2_eom


2026-10-25 01:11:32,167 INFO Ajustando HDBSCAN seed42_char_wb_2_4_min_df1_min_cluster_size2_min_samples5_eom


...

2026-10-25 02:51:01,981 INFO Ajustando HDBSCAN seed42_word_2_5_min_df10_min_cluster_size10_min_samples10_eom


2026-10-25 02:51:11,754 INFO Ajustando HDBSCAN seed42_word_2_5_min_df10_min_cluster_size20_min_samplesdefault_eom


2026-10-25 02:51:18,209 INFO Ajustando HDBSCAN seed42_word_2_5_min_df10_min_cluster_size20_min_samples2_eom


2026-10-25 02:51:25,136 INFO Ajustando HDBSCAN seed42_word_2_5_min_df10_min_cluster_size20_min_samples5_eom


2026-10-25 02:51:31,972 INFO Ajustando HDBSCAN seed42_word_2_5_min_df10_min_cluster_size20_min_samples10_eom


### Calcular a metrica composta e ordenar os resultados

A silhouette e reescalada para 0 a 1 e combinada com a proporcao de descricoes classificadas (`1 - noise_fraction`). O score composto determina a ordem; silhouette e ruido servem apenas de desempate quando os scores forem iguais.


In [7]:
if not 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING:
    raise ValueError(
        "Os limites devem atender a 1 <= MIN_CLUSTER_FOR_RANKING <= MAX_CLUSTERS_FOR_RANKING."
    )

if SILHOUETTE_WEIGHT < 0 or NOISE_WEIGHT < 0 or SILHOUETTE_WEIGHT + NOISE_WEIGHT <= 0:
    raise ValueError("Os pesos devem ser nao negativos e pelo menos um deles deve ser maior que zero.")

# Silhouette varia de -1 a 1; reescale para 0 a 1 para combinar com cobertura.
results["silhouette_quality"] = (results["silhouette_cosine_without_noise"] + 1) / 2
results["noise_quality"] = 1 - results["noise_fraction"]
weight_total = SILHOUETTE_WEIGHT + NOISE_WEIGHT
results["composite_score"] = (
    SILHOUETTE_WEIGHT * results["silhouette_quality"]
    + NOISE_WEIGHT * results["noise_quality"]
) / weight_total

# O score composto e o criterio principal; os outros criterios so desempataram scores iguais.
ranking = (
    results.loc[
        results["cluster_count_excluding_noise"].between(
            MIN_CLUSTER_FOR_RANKING, MAX_CLUSTERS_FOR_RANKING, inclusive="both",
        ) & results["silhouette_cosine_without_noise"].notna()
    ]
    .sort_values(
        ["composite_score", "silhouette_cosine_without_noise", "noise_fraction"],
        ascending=[False, False, True], na_position="last",
    )
    .reset_index(drop=True)
)
if ranking.empty:
    raise ValueError(
        f"Nenhum resultado atende ao intervalo de {MIN_CLUSTER_FOR_RANKING} a "
        f"{MAX_CLUSTERS_FOR_RANKING} clusters. Ajuste min_cluster_size/min_samples ou o limite."
    )
# Explicita a posicao de cada configuracao depois dos filtros e criterios acima.
ranking.insert(0, "rank", range(1, len(ranking) + 1))


In [8]:
display(ranking)

,rank,run_id,sample_seed,representation,analyzer,ngram_range,min_df,feature_count,zero_vector_count,min_cluster_size,...,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samplesdefault_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,5,...,False,736,9817,0.49085,20000,10183,0.518910,0.759455,0.50915,0.696879
1,2,seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samples5_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,5,...,False,736,9817,0.49085,20000,10183,0.518910,0.759455,0.50915,0.696879
2,3,seed42_word_2_5_min_df10_min_cluster_size20_min_samplesdefault_eom,42,word_2_5,word,"(2, 5)",10,930,10796,20,...,False,101,15999,0.79995,20000,4001,0.712737,0.856369,0.20005,0.692289
3,4,seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samples10_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,5,...,False,381,12053,0.60265,20000,7947,0.565642,0.782821,0.39735,0.686453
4,5,seed42_char_wb_5_10_min_df10_min_cluster_size10_min_samples2_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,10,...,False,554,7848,0.39240,20000,12152,0.424243,0.712122,0.60760,0.685991
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
404,405,seed42_char_wb_2_4_min_df2_min_cluster_size2_min_samples10_eom,42,char_wb_2_4,char_wb,"(2, 4)",2,23288,0,2,...,False,399,15816,0.79080,20000,4184,0.274880,0.637440,0.20920,0.530380
405,406,seed42_word_2_5_min_df1_min_cluster_size10_min_samples2_eom,42,word_2_5,word,"(2, 5)",1,100000,1046,10,...,False,538,9891,0.49455,20000,10109,0.076886,0.538443,0.50545,0.530195
406,407,seed42_word_2_5_min_df1_min_cluster_size10_min_samples5_eom,42,word_2_5,word,"(2, 5)",1,100000,1046,10,...,False,606,9597,0.47985,20000,10403,0.052305,0.526152,0.52015,0.524652
407,408,seed42_word_2_5_min_df2_min_cluster_size20_min_samples2_eom,42,word_2_5,word,"(2, 5)",2,20692,3690,20,...,False,125,11557,0.57785,20000,8443,0.090094,0.545047,0.42215,0.514323


In [9]:
ranking.head(40)

,rank,run_id,sample_seed,representation,analyzer,ngram_range,min_df,feature_count,zero_vector_count,min_cluster_size,...,frequency_weighted,cluster_count_excluding_noise,noise_count,noise_fraction,silhouette_base_sample_size,silhouette_sample_size,silhouette_cosine_without_noise,silhouette_quality,noise_quality,composite_score
0,1,seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samplesdefault_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,5,...,False,736,9817,0.49085,20000,10183,0.518910,0.759455,0.50915,0.696879
1,2,seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samples5_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,5,...,False,736,9817,0.49085,20000,10183,0.518910,0.759455,0.50915,0.696879
2,3,seed42_word_2_5_min_df10_min_cluster_size20_min_samplesdefault_eom,42,word_2_5,word,"(2, 5)",10,930,10796,20,...,False,101,15999,0.79995,20000,4001,0.712737,0.856369,0.20005,0.692289
3,4,seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samples10_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,5,...,False,381,12053,0.60265,20000,7947,0.565642,0.782821,0.39735,0.686453
4,5,seed42_char_wb_5_10_min_df10_min_cluster_size10_min_samples2_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,10,...,False,554,7848,0.39240,20000,12152,0.424243,0.712122,0.60760,0.685991
5,6,seed42_char_wb_5_10_min_df5_min_cluster_size5_min_samplesdefault_eom,42,char_wb_5_10,char_wb,"(5, 10)",5,31705,22,5,...,False,764,10613,0.53065,20000,9387,0.507340,0.753670,0.46935,0.682590
6,7,seed42_char_wb_5_10_min_df5_min_cluster_size5_min_samples5_eom,42,char_wb_5_10,char_wb,"(5, 10)",5,31705,22,5,...,False,764,10613,0.53065,20000,9387,0.507340,0.753670,0.46935,0.682590
7,8,seed42_char_wb_5_10_min_df10_min_cluster_size10_min_samples5_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,10,...,False,413,10137,0.50685,20000,9863,0.485829,0.742915,0.49315,0.680473
8,9,seed42_char_wb_5_10_min_df10_min_cluster_size10_min_samplesdefault_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,10,...,False,290,12167,0.60835,20000,7833,0.551087,0.775544,0.39165,0.679570
9,10,seed42_char_wb_5_10_min_df10_min_cluster_size10_min_samples10_eom,42,char_wb_5_10,char_wb,"(5, 10)",10,18111,47,10,...,False,290,12167,0.60835,20000,7833,0.551087,0.775544,0.39165,0.679570


## Salvar assigments de um teste específico

In [10]:
ASSIGNMENT_RUN_ID = ASSIGNMENT_RUN_ID or ranking.iloc[0]["run_id"]

run_assignments, run_cluster_summary = summarize_clusters(
    items_by_run[ASSIGNMENT_RUN_ID],
    labels_by_run[ASSIGNMENT_RUN_ID],
)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ%f")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)

assignment_path = output_dir / f"assignments_{ASSIGNMENT_RUN_ID}.csv"
if assignment_path.exists():
    raise FileExistsError(f"Arquivo já existe: {assignment_path}")

run_assignments.to_csv(assignment_path, index=False, encoding="utf-8-sig")
print(f"Assignments salvos em: {assignment_path}")

Assignments salvos em: C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tfidf_hdbscan\202501-202512_20261025T055138Z888595\assignments_seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samplesdefault_eom.csv


## 5. Revisar um resultado e salvar relat?rios

Escolha `SELECTED_RUN_ID` ap?s comparar quantidade de clusters, ru?do e silhouette. A tabela de resumo inclui `-1` como grupo de ru?do. Os arquivos s?o gravados em uma pasta com identificador UTC ?nico para preservar execu??es anteriores.

Os gr?ficos ficam em `tsne_plots/`, um PNG por configura??o, e as coordenadas da amostra em `tsne_coordinates/`, dentro da pasta da execu??o.


In [11]:
SELECTED_RUN_ID = ranking.iloc[0]["run_id"]
selected = results.set_index("run_id").loc[SELECTED_RUN_ID]
selected_items = items_by_run[SELECTED_RUN_ID]
selected_labels = labels_by_run[SELECTED_RUN_ID]
assignments, cluster_summary = summarize_clusters(selected_items, selected_labels)
cluster_summary = cluster_summary.sort_values("frequencia_total", ascending=False).reset_index(drop=True)
cluster_sizes = assignments.loc[assignments["cluster"].ne(-1)].groupby("cluster")[DESCRIPTION_NORMALIZED].size()
singleton_clusters = cluster_sizes[cluster_sizes.eq(1)].index
frequent_singletons = assignments[assignments["cluster"].isin(singleton_clusters)].sort_values(
    FREQUENCY, ascending=False
)
frequent_noise = assignments[assignments["cluster"].eq(-1)].sort_values(
    FREQUENCY, ascending=False
)
display(cluster_summary.head(20))
display(frequent_singletons.head(20))
display(frequent_noise.head(20))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = OUTPUT_ROOT / f"{PERIOD_LABEL}_{RUN_ID}"
output_dir.mkdir(parents=True, exist_ok=False)
results.to_csv(output_dir / "metrics.csv", index=False)
ranking.to_csv(output_dir / "ranking.csv", index=False)
assignments.to_csv(output_dir / "assignments.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(output_dir / "cluster_summary.csv", index=False, encoding="utf-8-sig")
frequent_singletons.to_csv(
    output_dir / "frequent_singleton_descriptions.csv", index=False, encoding="utf-8-sig"
)
frequent_noise.to_csv(
    output_dir / "frequent_noise_descriptions.csv", index=False, encoding="utf-8-sig"
)

,cluster,descricoes_unicas,frequencia_total
0,-1,9817,36428
1,537,6,3595
2,458,43,2531
3,534,19,1948
4,293,53,1735
5,314,19,1712
6,378,68,1665
7,229,35,1243
8,668,67,1229
9,734,35,1192


,descricao_normalizada,frequencia,cluster,ruido


,descricao_normalizada,frequencia,cluster,ruido
4904,ceia,567,-1,True
5079,chamex solution a4 c 10 resmas,442,-1,True
8014,etanol,350,-1,True
17292,saco lixo 50lts pct c 10und preto,264,-1,True
13943,pao doce com creme kg,246,-1,True
15321,polpa acai,230,-1,True
19894,vinho acai,223,-1,True
8152,farinha aveia,216,-1,True
5113,cheiro verde,204,-1,True
14141,papel a4 go office eco 500 fls fsc reciclado100 cucoc,190,-1,True


## Salvar um modelo pelo Run ID

Informe o `MODEL_RUN_ID` abaixo para exportar o TF-IDF, o HDBSCAN ajustado, suas configura??es e os assignments de treino. A exporta??o cria uma pasta nova em `models/tfidf_hdbscan/` para preservar modelos anteriores.


In [12]:
MODEL_RUN_ID = MODEL_RUN_ID or ranking.iloc[0]["run_id"]

if MODEL_RUN_ID not in labels_by_run:
    raise ValueError(f"Run ID nao encontrado. Opcoes: {list(labels_by_run)}")

model_run = results.set_index("run_id").loc[MODEL_RUN_ID]
model_items = items_by_run[MODEL_RUN_ID]
model_labels = labels_by_run[MODEL_RUN_ID]
model_representation = model_run["representation"]
analyzer, ngram_range = ANALYZER_CONFIGS[model_representation]

model_vectorizer = TfidfVectorizer(
    analyzer=analyzer,
    ngram_range=ngram_range,
    min_df=int(model_run["min_df"]),
    max_features=MAX_FEATURES, max_df=MAX_DF,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float64,
)
model_matrix = model_vectorizer.fit_transform(
    model_items[DESCRIPTION_NORMALIZED].astype(str)
)
model_min_samples = (
    None if pd.isna(model_run["min_samples"]) else int(model_run["min_samples"])
)
saved_model = HDBSCAN(
    min_cluster_size=int(model_run["min_cluster_size"]),
    min_samples=model_min_samples,
    metric="cosine",
    algorithm="brute",
    cluster_selection_method=model_run["cluster_selection_method"],
    n_jobs=HDBSCAN_N_JOBS,
    copy=True,
)
saved_model.fit(model_matrix)
if not np.array_equal(saved_model.labels_, model_labels):
    raise RuntimeError("Os labels do modelo refeito nao coincidem com os assignments do Run ID.")

model_assignments, _ = summarize_clusters(model_items, model_labels)
model_output_dir = MODEL_ROOT / f"{PERIOD_LABEL}_{RUN_ID}_{MODEL_RUN_ID}"
model_output_dir.mkdir(parents=True, exist_ok=False)
pd.to_pickle(model_vectorizer, model_output_dir / "tfidf_vectorizer.pkl")
pd.to_pickle(saved_model, model_output_dir / "hdbscan_model.pkl")
model_assignments.to_csv(
    model_output_dir / "training_assignments.csv", index=False, encoding="utf-8-sig"
)
model_config = {
    "run_id": MODEL_RUN_ID,
    "period": PERIOD_LABEL,
    "dataset_name": DATASET_NAME,
    "sample_seed": int(model_run["sample_seed"]),
    "representation": model_representation,
    "analyzer": analyzer,
    "ngram_range": list(ngram_range),
    "tfidf": {
        "min_df": int(model_run["min_df"]),
        "max_features": MAX_FEATURES, "max_df": MAX_DF, "dtype": "float64",
        "sublinear_tf": True,
        "norm": "l2",
    },
    "hdbscan": {
        "min_cluster_size": int(model_run["min_cluster_size"]),
        "min_samples": model_min_samples,
        "metric": "cosine",
        "algorithm": "brute",
        "cluster_selection_method": model_run["cluster_selection_method"],
    },
    "frequency_weighted": False,
    "artifacts": {
        "vectorizer": "tfidf_vectorizer.pkl",
        "model": "hdbscan_model.pkl",
        "training_assignments": "training_assignments.csv",
    },
}
(model_output_dir / "config.json").write_text(
    json.dumps(model_config, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Modelo do Run ID %s salvo em %s", MODEL_RUN_ID, model_output_dir)


2026-10-25 02:51:46,057 INFO Modelo do Run ID seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samplesdefault_eom salvo em C:\Users\samuel.simao\Sources\Nofis-Classifier\models\tfidf_hdbscan\202501-202512_20261025T055138Z_seed42_char_wb_5_10_min_df10_min_cluster_size5_min_samplesdefault_eom


## 6. Gerar visualiza??es t-SNE

Esta etapa roda depois do ajuste e do ranking. Ela projeta as matrizes TF-IDF j? calculadas e salva as coordenadas e os gr?ficos, sem executar novamente o HDBSCAN.


In [13]:
# Calcula as proje??es 2D existentes e uma proje??o 3D separada por representa??o.
tsne_by_representation = {}
tsne_3d_by_representation = {}
for (sample_seed, representation, min_df), matrix in matrices_by_representation.items():
    logger.info("Calculando t-SNE 2D para %s, min_df=%d, seed=%d", representation, min_df, sample_seed)
    tsne_indices, tsne_coordinates, tsne_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE,
    )
    tsne_by_representation[(sample_seed, representation, min_df)] = {
        "indices": tsne_indices, "coordinates": tsne_coordinates,
        "reduced_dimensions": tsne_dimensions,
    }

    logger.info("Calculando t-SNE 3D para %s, min_df=%d, seed=%d", representation, min_df, sample_seed)
    tsne_3d_indices, tsne_3d_coordinates, tsne_3d_dimensions = compute_tsne_projection(
        matrix, TSNE_MAX_POINTS, TSNE_PERPLEXITY, TSNE_MAX_ITER,
        TSNE_RANDOM_STATE, n_components=TSNE_3D_N_COMPONENTS,
    )
    if not np.array_equal(tsne_indices, tsne_3d_indices):
        raise ValueError("As proje??es 2D e 3D devem usar a mesma amostra de descri??es.")
    tsne_3d_by_representation[(sample_seed, representation, min_df)] = {
        "indices": tsne_3d_indices, "coordinates": tsne_3d_coordinates,
        "reduced_dimensions": tsne_3d_dimensions,
    }

# Salva coordenadas 2D e um PNG por configura??o.
tsne_coordinates_dir = output_dir / "tsne_coordinates"
tsne_plots_dir = output_dir / "tsne_plots"
tsne_3d_html_dir = output_dir / "tsne_3d_html"
tsne_coordinates_dir.mkdir()
tsne_plots_dir.mkdir()
tsne_3d_html_dir.mkdir()
tsne_artifacts = {}
for (sample_seed, representation, min_df), projection in tsne_by_representation.items():
    indices = projection["indices"]
    coordinates = projection["coordinates"]
    coordinate_frame = items_by_seed[sample_seed].iloc[indices][
        [DESCRIPTION_NORMALIZED, FREQUENCY]
    ].reset_index(drop=True)
    coordinate_frame["tsne_1"] = coordinates[:, 0]
    coordinate_frame["tsne_2"] = coordinates[:, 1]
    coordinate_name = f"seed{sample_seed}_{representation}_min_df{min_df}_coordinates.csv"
    coordinate_frame.to_csv(
        tsne_coordinates_dir / coordinate_name, index=False, encoding="utf-8-sig",
    )
    tsne_artifacts[f"seed{sample_seed}_{representation}_min_df{min_df}"] = {
        "coordinates_file": f"tsne_coordinates/{coordinate_name}",
        "sample_size": len(indices),
        "reduced_dimensions": projection["reduced_dimensions"],
    }

for run_id, labels in labels_by_run.items():
    result_row = results.loc[results["run_id"].eq(run_id)].iloc[0]
    representation_key = (
        int(result_row["sample_seed"]), result_row["representation"], int(result_row["min_df"]),
    )
    projection = tsne_by_representation[representation_key]
    projection_3d = tsne_3d_by_representation[representation_key]
    plot_labels = np.asarray(labels)[projection["indices"]]
    plot_path = tsne_plots_dir / f"{run_id}.png"
    save_tsne_plot(projection["coordinates"], plot_labels, run_id, plot_path)
    tsne_artifacts[f"plot:{run_id}"] = str(plot_path.relative_to(output_dir))

    plot_3d_path = tsne_3d_html_dir / f"{run_id}.html"
    save_tsne_3d_html(projection_3d["coordinates"], plot_labels, run_id, plot_3d_path)
    tsne_artifacts[f"plot_3d:{run_id}"] = str(plot_3d_path.relative_to(output_dir))

logger.info("Imagens t-SNE 2D e gr?ficos HTML 3D salvos: %d", len(labels_by_run))

run_metadata = {
    "comparison": comparison_metadata(items_by_seed[SAMPLE_SEEDS[0]]),
    "dataset_name": DATASET_NAME, "period": PERIOD_LABEL,
    "frequency_weighted": False,
    "dataset_path": str(DATASET_PATH), "max_descriptions": MAX_DESCRIPTIONS,
    "sample_seeds": SAMPLE_SEEDS, "analyzer_configs": {
        representation: {"analyzer": analyzer, "ngram_range": list(ngram_range)}
        for representation, (analyzer, ngram_range) in ANALYZER_CONFIGS.items()
    },
    "min_df_values": MIN_DF_VALUES, "max_features": MAX_FEATURES,
    "selected_run_id": SELECTED_RUN_ID,
    "min_cluster_for_ranking": MIN_CLUSTER_FOR_RANKING,
    "max_clusters_for_ranking": MAX_CLUSTERS_FOR_RANKING,
    "min_cluster_size_values": MIN_CLUSTER_SIZE_VALUES,
    "min_samples_values": MIN_SAMPLES_VALUES,
    "cluster_selection_method": CLUSTER_SELECTION_METHOD,
    "silhouette_sample_size": SILHOUETTE_SAMPLE_SIZE,
    "tsne": {
        "max_points": TSNE_MAX_POINTS, "perplexity": TSNE_PERPLEXITY,
        "max_iter": TSNE_MAX_ITER, "random_state": TSNE_RANDOM_STATE,
        "svd_components": TSNE_SVD_COMPONENTS,
        "three_dimensional_components": TSNE_3D_N_COMPONENTS,
        "artifacts": tsne_artifacts,
    },
}
(output_dir / "run.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
logger.info("Relatorios e imagens t-SNE salvos em %s", output_dir)


2026-10-25 02:51:46,100 INFO Calculando t-SNE 2D para char_wb_2_4, min_df=1, seed=42


2026-10-25 02:51:54,689 INFO Calculando t-SNE 3D para char_wb_2_4, min_df=1, seed=42


2026-10-25 02:52:19,015 INFO Calculando t-SNE 2D para char_wb_2_4, min_df=2, seed=42


2026-10-25 02:52:32,278 INFO Calculando t-SNE 3D para char_wb_2_4, min_df=2, seed=42


2026-10-25 02:53:05,812 INFO Calculando t-SNE 2D para char_wb_2_4, min_df=5, seed=42


...

2026-10-25 03:14:17,760 INFO Calculando t-SNE 3D para word_2_5, min_df=5, seed=42


2026-10-25 03:15:22,075 INFO Calculando t-SNE 2D para word_2_5, min_df=10, seed=42


2026-10-25 03:15:33,668 INFO Calculando t-SNE 3D para word_2_5, min_df=10, seed=42


2026-10-25 03:18:18,515 INFO Imagens t-SNE 2D e gr?ficos HTML 3D salvos: 640


2026-10-25 03:18:18,540 INFO Relatorios e imagens t-SNE salvos em C:\Users\samuel.simao\Sources\Nofis-Classifier\reports\tfidf_hdbscan\202501-202512_20261025T055138Z


## 7. Interpreta??o

- `noise_fraction` alta significa que muitas descri??es n?o entraram em grupos densos para os par?metros escolhidos.
- `min_cluster_size` controla o menor grupo selecion?vel. `min_samples` maior tende a exigir evid?ncia local mais forte e pode aumentar o ru?do.
- Silhouette mais alta deve ser lida junto da fra??o de ru?do e do tamanho da amostra usada para calcul?-la; uma pontua??o calculada em poucos itens pode ser enganosa.
- HDBSCAN n?o atribui automaticamente novos produtos a clusters neste fluxo e n?o define categorias sem revis?o humana.

- Os HTML em `tsne_3d_html/` cont?m uma proje??o t-SNE 3D que pode ser girada com o mouse e aberta sem depend?ncias externas.
